In [ ]:
!pip install supar openai sentencepiece wandb nltk spacy huggingface_hub rouge-score bert-score numpy language_tool_python==2.9.2 scikit-learn scipy tqdm bitsandbytes transformers Pillow ftfy regex flake8 yapf isort yacs gdown tb-nightly future wilds tabulate torch==2.5.1 torchvision==0.20.1 torchaudio==2.5.1 --extra-index-url https://download.pytorch.org/whl/cu124

In [ ]:
import os

# Specify the full path where you want to create the directory
path = "/kaggle/working/logs"

# Create the directory
os.makedirs(path, exist_ok=True)

print(f"Directory created at: {path}")

In [ ]:
#!/usr/bin/env python

import os
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True' # Add this early

import time, gc, re, json, random, string, heapq, logging, argparse, collections
import numpy as np
import nltk
from nltk.tokenize import word_tokenize, sent_tokenize
from nltk.tokenize.treebank import TreebankWordDetokenizer
from scipy.stats import entropy
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
# from rouge_score import rouge_scorer # Using custom ROUGE
from bert_score import score as bert_score
import difflib
from nltk.metrics.distance import edit_distance
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
import matplotlib.pyplot as plt
from huggingface_hub import login
import unicodedata

# Suppress Warnings
import warnings
warnings.filterwarnings("ignore")
from transformers import logging as hf_logging
hf_logging.set_verbosity_error()

from tqdm import tqdm

# External Libraries for Grammar Checking
# import spacy # Not actively used in the final version, can be removed if not needed for other purposes
# import language_tool_python # Not actively used for Urdu

# Urdu-specific imports
try:
    import stanza
    stanza.download('ur', verbose=False)  # Download Urdu model
except:
    print("Stanza not available, using fallback methods")

# Argument Parsing
parser = argparse.ArgumentParser(description='Multi-objective Genetic Search for Prompt Optimization - Urdu')
parser.add_argument('--mode', default="Instruction Only", help='Mode of instructions/prompts')
parser.add_argument('--num-shots', default=0, type=int, help='Number of examples in the prompt (set to 0 as per previous changes)') # Defaulting to 0
parser.add_argument('--batch-size', default=1, type=int, help='Batch size')
parser.add_argument('--task-idx', default=0, type=int, help='Index of the task')
parser.add_argument('--seed', default=3, type=int, help='Seed for sampling examples')
parser.add_argument('--train-seed', default=42, type=int, help='Seed for splitting and sampling edit operations')
parser.add_argument('--num-compose', default=1, type=int, help='Number of edits composed for one candidate')
parser.add_argument('--num-train', default=250, type=int, help='Number of examples in score set for GA evaluation')
parser.add_argument('--level', default="phrase", help='Level for edit operations')
parser.add_argument('--simulated-anneal', action='store_true', default=False, help='Use simulated anneal when candidate score <= base score')
parser.add_argument('--agnostic', action='store_true', default=False, help='Use template task-agnostic instruction')
parser.add_argument('--print-orig', action='store_true', default=True, help='Print original instruction and evaluate its performance')
parser.add_argument('--write-preds', action='store_true', default=True, help='Store predictions in a JSON file')
parser.add_argument('--meta-dir', default='/kaggle/working/logs/', help='Directory to store metadata')
parser.add_argument('--meta-name', default='urdu_llama_mutation_search.txt', help='Metadata filename')
parser.add_argument('--patience', default=5, type=int, help='Max patience')
parser.add_argument('--num-candidates', default=5, type=int, help='Number of candidates per iteration (used for mutation choices, not directly for population size)')
parser.add_argument('--num-iter', default=10, type=int, help='Maximum iterations')
parser.add_argument('--key-id', default=None, type=int, help='OpenAI key ID if applicable')
parser.add_argument('--edits', nargs="+", default=['del', 'swap', 'sub', 'add'], help='Edit operations')
parser.add_argument('--tournament-selection', default=2, type=int, help='Number of tournament selections')
parser.add_argument('--population-size', default=10, type=int, help='Population size for GA')
parser.add_argument('--num-offspring', default=0, type=int, help='Number of offspring per iteration (should be even for pair-wise crossover)')
parser.add_argument('--mutation-prob', default=0.5, type=float, help='Mutation probability')
parser.add_argument('--data-dir', default='/kaggle/input/urdu-xlsum/', help='Dataset directory')
parser.add_argument('--project-name', default='', help='WandB project name')
parser.add_argument('--budget', default=7000, type=int, help='API call budget')
args, unknown = parser.parse_known_args()

# Ensure meta_dir exists
os.makedirs(args.meta_dir, exist_ok=True)

# Clear score files at the start of the run
for fname in ['rouge_scores.txt', 'bert_scores.txt', 'bleu_scores.txt']:
    if os.path.exists(os.path.join(args.meta_dir, fname)):
        open(os.path.join(args.meta_dir, fname), 'w').close()
    else: # If file doesn't exist, create it
        with open(os.path.join(args.meta_dir, fname), 'w') as f:
            pass


# Initialize Stanza for Urdu
try:
    urdu_nlp = stanza.Pipeline('ur', processors='tokenize,pos,lemma,depparse', use_gpu=False, verbose=False)
    print("Stanza Urdu pipeline initialized successfully")
except Exception as e:
    urdu_nlp = None
    print(f"Stanza not available or failed to initialize: {e}, using fallback tokenization")

# Initialize progress bar
global_progress_bar = tqdm(total=args.budget, desc="API Calls", leave=False, position=0, dynamic_ncols=True)

try:
    import wandb
    wandb.login(key='') # Replace with your key or use environment variable
    wandb.init(project=args.project_name)
    wandb.config.update(args)
    tqdm.write("Wandb is setup\n")
except Exception as e:
    tqdm.write(f"Wandb initialization error: {e}\n")

# Handle Hugging Face token
hf_token = "" # Replace with your token or use environment variable
if not hf_token:
    raise ValueError("Hugging Face token is required for gated model access. Provide via --hf-token or set HF_TOKEN environment variable.")
try:
    login(hf_token)
    tqdm.write("Successfully logged in to Hugging Face Hub")
except Exception as e:
    raise ValueError(f"Failed to authenticate with Hugging Face: {str(e)}")

# Model Setup (Llama 3.1 8B Instruct)
from transformers import pipeline, AutoTokenizer
import torch
import gc

# Set random seed for reproducibility
torch.random.manual_seed(args.seed) # Use args.seed for torch seed as well
np.random.seed(args.seed)
random.seed(args.seed)


# Model name
model_name = "meta-llama/Meta-Llama-3.1-8B-Instruct"

# Initialize pipeline with bfloat16 and multi-GPU support
try:
    pipe = pipeline(
        "text-generation",
        model=model_name,
        model_kwargs={"torch_dtype": torch.bfloat16},
        device_map="auto",
        token=hf_token
    )
except RuntimeError as e:
    if "CUDA out of memory" in str(e):
        print("CUDA out of memory during pipeline initialization, clearing cache and retrying...")
        torch.cuda.empty_cache()
        gc.collect()
        pipe = pipeline(
            "text-generation",
            model=model_name,
            model_kwargs={"torch_dtype": torch.bfloat16},
            device_map="auto",
            token=hf_token
        )
    else:
        raise e

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(
    model_name,
    token=hf_token,
    trust_remote_code=True
)

# Generation arguments
generation_args = {
    "max_new_tokens": 60, # Max tokens for the generated summary
    "temperature": 0.1,
    "do_sample": False
}
MAX_ARTICLE_TOKENS = 1500 # Max tokens for the input article after truncation

# Verify GPU utilization
if torch.cuda.is_available():
    print("GPU Utilization:")
    for i in range(torch.cuda.device_count()):
        print(f"GPU {i}: {torch.cuda.memory_allocated(i) / 1024**3:.2f} GB allocated, "
              f"{torch.cuda.memory_reserved(i) / 1024**3:.2f} GB reserved")
else:
    print("CUDA not available. Running on CPU.")

# Initialize Evaluation cache
evaluation_cache = {}

# Urdu-specific utility functions
def is_urdu_text(text):
    """Check if text contains Urdu characters"""
    if not isinstance(text, str): return False
    urdu_range = range(0x0600, 0x06FF)  # Arabic/Urdu Unicode range
    return any(ord(char) in urdu_range for char in text)

def enhanced_urdu_tokenize(text):
    """Enhanced Urdu tokenization with better handling"""
    if not isinstance(text, str) or not text.strip():
        return []

    if urdu_nlp:
        try:
            doc = urdu_nlp(text)
            tokens = []
            for sentence in doc.sentences:
                for word in sentence.words:
                    # Skip empty tokens
                    if word.text.strip():
                        tokens.append(word.text)
            return tokens
        except Exception:
            pass

    # Enhanced fallback tokenization
    # Handle Urdu text with better regex
    tokens = re.findall(r'[\u0600-\u06FF\u0750-\u077F]+|[^\s\u0600-\u06FF\u0750-\u077F]+', text)
    return [token.strip() for token in tokens if token.strip()]

def urdu_tokenize(text):
    """Tokenize Urdu text using enhanced method"""
    return enhanced_urdu_tokenize(text)

def urdu_sent_tokenize(text):
    """Sentence tokenization for Urdu text"""
    if not isinstance(text, str): return []
    if urdu_nlp:
        try:
            doc = urdu_nlp(text)
            return [sentence.text for sentence in doc.sentences]
        except Exception as e:
            pass
    sentences = re.split(r'([۔؟!])', text)
    result = []
    current_sentence = ""
    for part in sentences:
        current_sentence += part
        if part in '۔؟!':
            result.append(current_sentence.strip())
            current_sentence = ""
    if current_sentence.strip():
        result.append(current_sentence.strip())
    return [s for s in result if s]


def urdu_detokenize(tokens):
    """Join Urdu tokens back into text"""
    if not tokens:
        return ""
    result = tokens[0]
    for token in tokens[1:]:
        if not re.match(r'^[۔؟!،؍]', token) and not unicodedata.category(token[0]).startswith('M'):
            result += " " + token
        else:
            result += token
    return result

def is_meaningful_phrase(phrase_text: str, is_single_word: bool = False) -> bool:
    """
    Checks if an Urdu phrase is meaningful.
    """
    if not phrase_text or not phrase_text.strip():
        return False

    phrase_cleaned = phrase_text.strip()
    tokens = enhanced_urdu_tokenize(phrase_cleaned) # Assumes enhanced_urdu_tokenize is defined

    if not tokens:
        return False

    # Basic length checks for Urdu
    if is_single_word:
        if len(tokens) != 1: return False
        if len(phrase_cleaned) < 1: return False # Urdu words can be short
    else: # Multi-word phrases
        if len(tokens) < 2: return False
        if len(phrase_cleaned) < 3: return False # e.g., "کا میں"

    # Ensure it's predominantly Urdu characters
    urdu_char_count = sum(1 for char in phrase_cleaned if '\u0600' <= char <= '\u06FF')
    total_chars_no_space = len(phrase_cleaned.replace(" ", ""))
    if total_chars_no_space == 0: return False
    if urdu_char_count == 0 and total_chars_no_space > 0 : return False
    if urdu_char_count > 0 and urdu_char_count / total_chars_no_space < 0.5: return False

    single_stop_words_ur = {}
    multi_word_stop_phrases_ur = {}

    if is_single_word and phrase_cleaned in single_stop_words_ur:
        return False
    if not is_single_word and phrase_cleaned in multi_word_stop_phrases_ur:
        return False

    if not is_single_word: # For multi-word, ensure not ALL tokens are stop words
        if all(token in single_stop_words_ur for token in tokens):
            return False

    digit_count = sum(1 for char in phrase_cleaned if char.isdigit())
    if total_chars_no_space > 0 and (digit_count == total_chars_no_space): # All digits
        return False

    is_only_symbols = True
    if total_chars_no_space > 0:
        for char_symbol_check in phrase_cleaned:
            if char_symbol_check.isspace():
                continue
            if char_symbol_check.isalnum() or ('\u0600' <= char_symbol_check <= '\u06FF'):
                is_only_symbols = False
                break
        if is_only_symbols:
            return False

    return True

def _gather_all_dependents(
    head_word_obj,
    sentence_words_list,
    consumed_word_ids,
    max_depth=5
):
    """
    Gathers all direct and indirect dependents of a head word that have not yet been consumed.
    This forms a single, coherent syntactic unit (a simulated constituent).
    """
    phrase_words = {head_word_obj}
    queue = collections.deque([(head_word_obj, 0)])
    visited_word_ids = {head_word_obj.id}

    while queue:
        current_word, depth = queue.popleft()

        if depth >= max_depth:
            continue

        # Find all words that depend on the current_word
        for potential_dependent in sentence_words_list:
            if potential_dependent.head == current_word.id and \
               potential_dependent.id not in visited_word_ids and \
               potential_dependent.id not in consumed_word_ids:

                phrase_words.add(potential_dependent)
                visited_word_ids.add(potential_dependent.id)
                queue.append((potential_dependent, depth + 1))

    return phrase_words

def get_constituency_like_phrases_for_urdu(instruction: str) -> list:
    """
    Extracts non-overlapping, constituency-like phrases from Urdu text using
    Stanza's dependency parse. This is the new, enhanced mechanism.
    """
    global urdu_nlp # Assuming urdu_nlp is a global Stanza pipeline object
    if not urdu_nlp or not isinstance(instruction, str) or not instruction.strip():
        return []

    final_phrases = []
    try:
        doc = urdu_nlp(instruction)
    except Exception as e:
        tqdm.write(f"Stanza processing error in phrase extraction: {e}. Returning empty list.")
        return []

    for sentence in doc.sentences:
        sentence_words = sentence.words
        consumed_word_ids = set()

        # We iterate through words as potential heads. The order can matter.
        # Iterating from right to left can sometimes help identify larger phrases first in Urdu.
        for head_word in reversed(sentence_words):
            if head_word.id in consumed_word_ids:
                continue

            # We only consider major content words as potential heads of multi-word phrases
            if head_word.upos not in ['NOUN', 'PROPN', 'VERB', 'ADJ']:
                continue

            # Gather all dependents of this head to form a potential phrase
            phrase_word_objects = _gather_all_dependents(
                head_word, sentence_words, consumed_word_ids
            )

            # We only form a multi-word phrase if the head gathered dependents
            if len(phrase_word_objects) > 1:
                # Sort the words by their original position in the sentence
                sorted_phrase_words = sorted(list(phrase_word_objects), key=lambda w: w.id)
                phrase_text = urdu_detokenize([w.text for w in sorted_phrase_words])

                if is_meaningful_phrase(phrase_text, is_single_word=False):
                    final_phrases.append(phrase_text)
                    # CRUCIAL: Mark all words in this phrase as consumed
                    for word in phrase_word_objects:
                        consumed_word_ids.add(word.id)

        # After finding all multi-word phrases, add any remaining unconsumed
        # single words that are meaningful on their own.
        for word in sentence_words:
            if word.id not in consumed_word_ids:
                if is_meaningful_phrase(word.text, is_single_word=True):
                    final_phrases.append(word.text)

    # Final deduplication and sorting (longer phrases first)
    unique_phrases = sorted(list(set(final_phrases)), key=len, reverse=True)

    if not unique_phrases:
        return []

    final_non_overlapping_phrases = []
    unique_phrases.sort(key=lambda p: len(enhanced_urdu_tokenize(p)), reverse=True)

    for phrase in unique_phrases:
        is_sub_phrase = False
        for existing_phrase in final_non_overlapping_phrases:
            # Check if the smaller phrase is a substring of the larger one
            if phrase in existing_phrase:
                is_sub_phrase = True
                break
        if not is_sub_phrase:
            final_non_overlapping_phrases.append(phrase)

    return final_non_overlapping_phrases

def get_urdu_phrases(instruction):
   """
   Top-level function to get phrases. This now calls the new constituency-like method.
   """
   return get_constituency_like_phrases_for_urdu(instruction)


def get_urdu_phrase_lookup(base_candidate):
    """Get phrase lookup dictionary for Urdu text"""
    phrases = get_urdu_phrases(base_candidate)
    phrase_lookup = {i: phrase for i, phrase in enumerate(phrases)}
    return phrase_lookup

def normalize_urdu_text(text):
    """Normalize Urdu text for better comparison"""
    if not text: return ""
    text = str(text).strip()
    text = unicodedata.normalize("NFKC", text)
    text = re.sub(r"\s+", " ", text)
    text = re.sub(r"[۔،؍؎؏ؘؙؚؐؑؒؓؔؕؖؗ؛؜]", "", text) # Keep Urdu specific punctuation if needed for ROUGE
    return text.strip()

def calculate_urdu_rouge(reference, generated):
    """Calculate ROUGE scores with proper Urdu text normalization"""
    ref_normalized = normalize_urdu_text(reference)
    gen_normalized = normalize_urdu_text(generated)
    ref_tokens = ref_normalized.split()
    gen_tokens = gen_normalized.split()

    if not ref_tokens or not gen_tokens:
        return {"rouge1": 0.0, "rouge2": 0.0, "rougeL": 0.0}

    ref_unigrams = set(ref_tokens)
    gen_unigrams = set(gen_tokens)
    if len(gen_unigrams) == 0 or len(ref_unigrams) == 0:
        rouge1_precision, rouge1_recall, rouge1 = 0.0, 0.0, 0.0
    else:
        rouge1_overlap = len(ref_unigrams.intersection(gen_unigrams))
        rouge1_precision = rouge1_overlap / len(gen_unigrams)
        rouge1_recall = rouge1_overlap / len(ref_unigrams)
        rouge1 = (2 * rouge1_precision * rouge1_recall / (rouge1_precision + rouge1_recall)) if (rouge1_precision + rouge1_recall) > 0 else 0.0

    ref_bigrams = set(zip(ref_tokens, ref_tokens[1:])) if len(ref_tokens) > 1 else set()
    gen_bigrams = set(zip(gen_tokens, gen_tokens[1:])) if len(gen_tokens) > 1 else set()
    if len(gen_bigrams) == 0 or len(ref_bigrams) == 0:
        rouge2_precision, rouge2_recall, rouge2 = 0.0, 0.0, 0.0
    else:
        rouge2_overlap = len(ref_bigrams.intersection(gen_bigrams))
        rouge2_precision = rouge2_overlap / len(gen_bigrams)
        rouge2_recall = rouge2_overlap / len(ref_bigrams)
        rouge2 = (2 * rouge2_precision * rouge2_recall / (rouge2_precision + rouge2_recall)) if (rouge2_precision + rouge2_recall) > 0 else 0.0

    def lcs_length(X, Y):
        m, n = len(X), len(Y)
        dp = [[0] * (n + 1) for _ in range(m + 1)]
        for i in range(1, m + 1):
            for j in range(1, n + 1):
                if X[i - 1] == Y[j - 1]:
                    dp[i][j] = dp[i - 1][j - 1] + 1
                else:
                    dp[i][j] = max(dp[i - 1][j], dp[i][j - 1])
        return dp[m][n]

    lcs_len = lcs_length(ref_tokens, gen_tokens)
    if len(gen_tokens) == 0 or len(ref_tokens) == 0:
        rougeL_precision, rougeL_recall, rougeL = 0.0, 0.0, 0.0
    else:
        rougeL_precision = lcs_len / len(gen_tokens)
        rougeL_recall = lcs_len / len(ref_tokens)
        rougeL = (2 * rougeL_precision * rougeL_recall / (rougeL_precision + rougeL_recall)) if (rougeL_precision + rougeL_recall) > 0 else 0.0
    return {"rouge1": rouge1, "rouge2": rouge2, "rougeL": rougeL}

def get_urdu_grammar_score(text):
    """Get grammar score for Urdu text using LLM with improved prompt"""
    if not is_urdu_text(text) or len(text.strip()) < 5:
        return 0

    # English System Prompt
    system_prompt = (
        "You are an expert Urdu grammar checker. Your task is to evaluate the "
        "grammar of the provided Urdu text and assign a numerical score from 0 to 100.\n"
        "Evaluation Criteria:\n"
        "- Sentence structure and syntax\n"
        "- Correct word usage\n"
        "- Punctuation and grammatical correctness\n"
        "- Language fluency and flow\n\n"
        "Return ONLY the numerical score, with no additional text or explanation."
    )

    # English User Prompt
    user_prompt = (
        "Analyze the grammar of the following Urdu text in detail:\n\n"
        f'Text: "{text}"\n\n'
        "Use the following scale for scoring:\n"
        "0-30: Very poor grammar, fundamental errors\n"
        "31-50: Weak grammar, several errors\n"
        "51-70: Average grammar, some errors\n"
        "71-85: Good grammar, minor errors\n"
        "86-100: Excellent grammar, no or negligible errors\n\n"
        "Score:"
    )

    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt},
    ]

    try:
        raw_output = complete_phi4(messages, max_tokens=5)
        # More robust number extraction
        numbers = re.findall(r"\b(\d{1,3})\b", raw_output)
        if numbers:
            score = int(numbers[0])
            return max(0, min(100, score))  # Clamp between 0-100
        return urdu_grammar_fallback(text)
    except Exception as e:
        tqdm.write(f"Grammar scoring error: {e}")
        return urdu_grammar_fallback(text)

def urdu_grammar_fallback(text):
    """Fallback grammar scoring for Urdu text"""
    score = 100
    if not is_urdu_text(text): score -= 50
    sentences = urdu_sent_tokenize(text)
    if not sentences: score -= 30
    else:
        if len(urdu_tokenize(text)) < 3: score -= 25
    if not any(ending in text for ending in ['۔', '؟', '!']): score -= 20
    for sentence in sentences:
        tokens = urdu_tokenize(sentence)
        if len(tokens) < 2 and len(sentences) > 1 : score -= 10
        elif len(tokens) > 40: score -= 15
    return max(0, score)

def clean_generated_paraphrase(generated_text, original_phrase):
    """Clean and post-process the generated paraphrase"""
    if not generated_text:
        return original_phrase

    # Remove common prefixes/suffixes that LLM might add
    prefixes_to_remove = [
        "متبادل عبارت:", "متبادل:", "جواب:", "answer:",
        "paraphrase:", "alternative:", "یہ ہے:", "یہ:"
    ]

    suffixes_to_remove = ["۔", ".", "!", "؟", "?"]

    cleaned = generated_text.strip()

    # Remove prefixes
    for prefix in prefixes_to_remove:
        if cleaned.lower().startswith(prefix.lower()):
            cleaned = cleaned[len(prefix):].strip()

    # Remove quotes
    cleaned = cleaned.strip('\'"\"\'')

    # Remove suffixes (punctuation)
    for suffix in suffixes_to_remove:
        if cleaned.endswith(suffix):
            cleaned = cleaned[:-len(suffix)].strip()

    # --- NEW FIX: Strip the invalid Unicode replacement character ---
    cleaned = cleaned.replace('\uFFFD', '').strip()

    # Remove extra whitespace
    cleaned = re.sub(r'\s+', ' ', cleaned).strip()

    return cleaned if cleaned else original_phrase

def is_valid_paraphrase(paraphrase, original_phrase, full_context):
    """Validate if the paraphrase is suitable"""
    if not paraphrase or not paraphrase.strip():
        return False

    # --- NEW FIX ---
    # Explicitly reject paraphrases containing the Unicode replacement character.
    if '\uFFFD' in paraphrase:
        tqdm.write(f"Validation failed: Paraphrase '{paraphrase}' contains invalid characters.")
        return False
    # --- END OF FIX ---

    # Check if it's just the original phrase
    if paraphrase.strip().lower() == original_phrase.strip().lower():
        return False

    # Check if it contains Urdu text
    if not is_urdu_text(paraphrase):
        return False

    # Check length constraints
    original_tokens = urdu_tokenize(original_phrase)
    paraphrase_tokens = urdu_tokenize(paraphrase)

    # Paraphrase shouldn't be too long or too short
    if len(paraphrase_tokens) > len(original_tokens) * 2.5 or len(paraphrase_tokens) < max(1, len(original_tokens) // 2):
        return False

    # Check if paraphrase contains the original phrase (LLM sometimes includes it)
    if original_phrase.lower() in paraphrase.lower() and paraphrase.lower() != original_phrase.lower():
        return False

    # Check if it looks like a complete sentence when it should be a phrase
    if len(original_tokens) <= 3 and any(punct in paraphrase for punct in ['۔', '؟', '!']):
        return False

    return True

def substitute_urdu_phrase(candidate, phrase_to_replace):
    """
    Substitute Urdu phrase with paraphrase using improved prompting and post-processing.
    Returns: (new_candidate_text, actual_paraphrase_used, original_phrase_to_replace)
    """
    if not phrase_to_replace.strip():
        return candidate, phrase_to_replace, phrase_to_replace

    # Check if phrase exists in candidate
    if phrase_to_replace not in candidate:
        return candidate, phrase_to_replace, phrase_to_replace

    # English System Prompt
    system_prompt = (
        "You are an expert in Urdu linguistics, specializing in synonyms and paraphrasing. "
        "Your task is to provide a suitable and improved alternative for a given Urdu phrase, "
        "considering its context.\n\n"
        "Key Instructions:\n"
        "- Provide ONLY the alternative phrase. Do not add any explanations or introductory text like 'Here is the alternative:'.\n"
        "- The alternative must preserve the original meaning of the phrase.\n"
        "- Ensure the new phrase is grammatically correct and maintains the fluency of the sentence.\n"
        "- Keep the response concise and clear."
    )

    # English User Prompt
    user_prompt = (
        f'Full Sentence Context: "{candidate}"\n\n'
        "Find a better, more suitable alternative for the following phrase from the sentence above:\n"
        f'Phrase to Replace: "{phrase_to_replace}"\n\n'
        "Alternative Phrase:"
    )

    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt},
    ]

    try:
        # Calculate appropriate max tokens based on original phrase length
        original_tokens = len(urdu_tokenize(phrase_to_replace))
        max_tokens = min(max(original_tokens + 10, 15), 60)

        generated_text = complete_phi4(messages, max_tokens=max_tokens)

        # Post-processing: Clean the generated text
        paraphrase = clean_generated_paraphrase(
            generated_text, phrase_to_replace
        )

        if is_valid_paraphrase(paraphrase, phrase_to_replace, candidate):
            # Perform substitution
            new_candidate = candidate.replace(phrase_to_replace, paraphrase, 1)

            # Validate grammar of new candidate
            grammar_score = get_urdu_grammar_score(new_candidate)
            if grammar_score >= 35:  # Slightly higher threshold
                return new_candidate, paraphrase, phrase_to_replace
            else:
                tqdm.write(
                    f"Grammar check failed for substitution. Score: {grammar_score}"
                )
                return candidate, phrase_to_replace, phrase_to_replace
        else:
            return candidate, phrase_to_replace, phrase_to_replace

    except Exception as e:
        tqdm.write(f"Error during Urdu paraphrasing: {e}")
        return candidate, phrase_to_replace, phrase_to_replace

def delete_urdu_phrase(candidate, phrase):
    """Delete Urdu phrase from candidate"""
    if not phrase.strip(): return candidate
    patterns = [r'\s+' + re.escape(phrase) + r'\s+', r'\s+' + re.escape(phrase), re.escape(phrase) + r'\s+', re.escape(phrase)]
    new_candidate = candidate
    for i, pat_str in enumerate(patterns):
        replacement = ' ' if i == 0 else ''
        temp_candidate, num_subs = re.subn(pat_str, replacement, new_candidate, 1)
        if num_subs > 0:
            new_candidate = temp_candidate
            break
    return ' '.join(new_candidate.split())

def add_urdu_phrase(candidate, phrase_to_add, after_phrase):
    """Add Urdu phrase to candidate after specified phrase or at the beginning"""
    if not phrase_to_add.strip(): return candidate
    if not after_phrase.strip() or after_phrase not in candidate:
        return phrase_to_add + " " + candidate
    else:
        return candidate.replace(after_phrase, after_phrase + " " + phrase_to_add, 1)

def swap_urdu_phrases(candidate, phrase_1, phrase_2):
    """Swap two Urdu phrases in candidate, now with overlap protection."""
    if not phrase_1.strip() or not phrase_2.strip() or phrase_1 == phrase_2:
        return candidate

    idx1 = candidate.find(phrase_1)
    idx2 = candidate.find(phrase_2)

    if idx1 == -1 or idx2 == -1:
        return candidate

    end1 = idx1 + len(phrase_1)
    end2 = idx2 + len(phrase_2)

    if max(idx1, idx2) < min(end1, end2):
        tqdm.write(
            f"Swap aborted: Phrases overlap. P1='{phrase_1}', P2='{phrase_2}'"
        )
        return candidate

    placeholder1 = "___PLACEHOLDER_SWAP_1___" + str(random.randint(1000, 9999))
    placeholder2 = "___PLACEHOLDER_SWAP_2___" + str(random.randint(1000, 9999))

    temp_candidate = candidate
    if idx1 < idx2:
        temp_candidate = temp_candidate.replace(phrase_1, placeholder1, 1)
        temp_candidate = temp_candidate.replace(phrase_2, placeholder2, 1)
    else:
        temp_candidate = temp_candidate.replace(phrase_2, placeholder2, 1)
        temp_candidate = temp_candidate.replace(phrase_1, placeholder1, 1)

    final_candidate = temp_candidate.replace(placeholder1, phrase_2, 1)
    final_candidate = final_candidate.replace(placeholder2, phrase_1, 1)

    return final_candidate


def perform_urdu_edit(edit, base_text, phrase_list_full, deleted_phrases_history):
    """Perform edit operation on Urdu text.
    Returns: (mutated_text, new_deleted_phrases_history, edit_description_string)
    """
    mutated = base_text
    phrase_list = list(phrase_list_full)
    edit_description = f"Attempted {edit}: "
    if edit == 'del':
        if not phrase_list:
            edit_description += "No matching Urdu phrase found for deletion."
            return base_text, deleted_phrases_history, edit_description
        chosen = random.choice(phrase_list)
        mutated = delete_urdu_phrase(base_text, chosen)
        if mutated != base_text:
            if chosen not in deleted_phrases_history: deleted_phrases_history.append(chosen)
            edit_description += f"Deleted phrase '{chosen}'."
        else: edit_description += f"Failed to delete phrase '{chosen}'."
    elif edit == 'swap':
        if len(phrase_list) < 2:
            edit_description += "Not enough matching Urdu phrases for swap."
            return base_text, deleted_phrases_history, edit_description
        p1, p2 = random.sample(phrase_list, 2)
        mutated = swap_urdu_phrases(base_text, p1, p2)
        if mutated != base_text: edit_description += f"Swapped '{p1}' with '{p2}'."
        else: edit_description += f"Failed to swap '{p1}' and '{p2}'."
    elif edit == 'sub':
        if not phrase_list:
            edit_description += "No matching Urdu phrase for substitution."
            return base_text, deleted_phrases_history, edit_description
        chosen_phrase_to_replace = random.choice(phrase_list)
        mutated, paraphrase_used, original_phrase = substitute_urdu_phrase(base_text, chosen_phrase_to_replace)
        if mutated != base_text and paraphrase_used != original_phrase:
            edit_description += f"Substituted '{original_phrase}' with '{paraphrase_used}'."
        elif paraphrase_used == original_phrase and mutated != base_text :
             edit_description += f"Replaced '{original_phrase}' with an identical phrase (formatting change)."
        elif paraphrase_used == original_phrase :
            edit_description += f"Substitution of '{original_phrase}' resulted in no change or was reverted."
        else:
            edit_description += f"Substitution of '{original_phrase}' with '{paraphrase_used}' was reverted."
    elif edit == 'add':
        if not deleted_phrases_history:
            edit_description += "No deleted Urdu phrases for addition."
            return base_text, deleted_phrases_history, edit_description
        phrase_to_add = random.choice(deleted_phrases_history)
        after = ""
        if phrase_list:
            after = random.choice(phrase_list)
            edit_description += f"Attempting to add '{phrase_to_add}' after '{after}'."
        else: edit_description += f"Attempting to prepend '{phrase_to_add}'."
        mutated = add_urdu_phrase(base_text, phrase_to_add, after)
        if mutated != base_text:
             if phrase_to_add in deleted_phrases_history: deleted_phrases_history.remove(phrase_to_add)
             edit_description = edit_description.replace("Attempting to add", "Successfully added")
        else: edit_description += " Addition resulted in no change."
    return mutated, deleted_phrases_history, edit_description

def safe_urdu_mutation(edit, base_text, phrase_list, deleted_phrases_history, grammar_threshold=30, meta_file_handle=None, mutation_step_info=""):
    """Safely perform Urdu mutation with grammar checking"""
    mutated, new_del_history, edit_desc = perform_urdu_edit(edit, base_text, phrase_list, deleted_phrases_history)
    full_edit_log = f"    {mutation_step_info} - Op: {edit} - Detail: {edit_desc}"
    if mutated == base_text:
        log_entry = f"{full_edit_log} -> No change to candidate."
        tqdm.write(log_entry)
        if meta_file_handle and not meta_file_handle.closed: meta_file_handle.write(log_entry + "\n")
        return base_text, deleted_phrases_history
    gscore = get_urdu_grammar_score(mutated)
    if gscore >= grammar_threshold:
        log_entry = f"{full_edit_log} -> Accepted. New Grammar: {gscore}. Candidate: '{mutated[:70]}...'"
    else:
        log_entry = f"{full_edit_log} -> Rejected. Low Grammar: {gscore}. Reverting to: '{base_text[:70]}...'"
        mutated = base_text # Revert
        new_del_history = deleted_phrases_history # Revert history
    tqdm.write(log_entry)
    if meta_file_handle and not meta_file_handle.closed: meta_file_handle.write(log_entry + "\n")
    return mutated, new_del_history

# Instruction Encoding Functions for Urdu
def encode_urdu_instruction(task, instruction_structure=['Definition'], number_of_examples=0, number_of_instances=100, null_word=None, data_seed=0, modified={}, args=args):
    random.seed(data_seed)
    with open(os.path.join(args.data_dir, task), 'r', encoding='utf-8') as json_file:
        data = json.load(json_file)
    instances = data["Instances"]
    num_available_instances = len(instances)
    instance_indices = list(range(num_available_instances))
    random.seed(args.seed)
    test_indices = random.sample(instance_indices, min(number_of_instances, num_available_instances))
    generic_instruction = ''
    current_definition = data.get("Definition", [""])[0]
    if 'Definition' in modified:
        current_definition = modified['Definition']
    if current_definition and current_definition.strip() != '-':
        generic_instruction = "Definition: " + current_definition.strip()
    promptlist = []
    answerlist = []
    for i in test_indices:
        raw_instance_input = instances[i]['input']
        instance_id = instances[i].get('id', f'test_idx_{i}')
        initial_token_ids = tokenizer.encode(raw_instance_input, add_special_tokens=False)
        original_token_length = len(initial_token_ids)
        if original_token_length <= MAX_ARTICLE_TOKENS:
            instance_input = raw_instance_input
        else:
            sentences = urdu_sent_tokenize(raw_instance_input)
            if not sentences:
                # tqdm.write(f"Warning: Sentence tokenization failed for ID {instance_id} (encode_urdu_instruction). Using hard token truncation.")
                truncated_token_ids = initial_token_ids[:MAX_ARTICLE_TOKENS]
                instance_input = tokenizer.decode(truncated_token_ids, skip_special_tokens=True)
            else:
                selected_sentences = []
                current_total_tokens = 0
                for sentence_text in sentences:
                    if not sentence_text.strip(): continue
                    sentence_tokens_llm = tokenizer.encode(sentence_text, add_special_tokens=False)
                    if current_total_tokens + len(sentence_tokens_llm) <= MAX_ARTICLE_TOKENS:
                        selected_sentences.append(sentence_text)
                        current_total_tokens += len(sentence_tokens_llm)
                    else:
                        if not selected_sentences and len(sentence_tokens_llm) > MAX_ARTICLE_TOKENS:
                            # tqdm.write(f"Warning: First sentence ({len(sentence_tokens_llm)} tokens) for ID {instance_id} (encode_urdu_instruction) exceeds MAX_ARTICLE_TOKENS ({MAX_ARTICLE_TOKENS}). Truncating it.")
                            truncated_first_sentence_ids = sentence_tokens_llm[:MAX_ARTICLE_TOKENS]
                            selected_sentences.append(tokenizer.decode(truncated_first_sentence_ids, skip_special_tokens=True))
                        break
                if not selected_sentences:
                    # tqdm.write(f"Warning: No sentences selected for ID {instance_id} (encode_urdu_instruction) after truncation. Using hard token truncation.")
                    truncated_token_ids = initial_token_ids[:MAX_ARTICLE_TOKENS]
                    instance_input = tokenizer.decode(truncated_token_ids, skip_special_tokens=True)
                else:
                    instance_input = " ".join(selected_sentences)
            final_truncated_len = len(tokenizer.encode(instance_input, add_special_tokens=False))
            # if original_token_length > MAX_ARTICLE_TOKENS :
            #      tqdm.write(f"ID {instance_id} (encode_urdu_instruction): Original tokens: {original_token_length}, Truncated to: {final_truncated_len} tokens (target <={MAX_ARTICLE_TOKENS}).")
        user_content = generic_instruction + "\n" + "Input: " + instance_input + "\nخلاصہ:"
        system_message_content = ""
        promptlist.append([{"role": "system", "content": system_message_content}, {"role": "user", "content": user_content}])
        answerlist.append(instances[i]['output'][0] if isinstance(instances[i]['output'], list) else instances[i]['output'])
    return promptlist, answerlist, test_indices

def training_encode_urdu_instruction(task, instruction_structure=['Definition'], number_of_examples=0, number_of_instances=100, null_word=None, data_seed=0, modified={}, args=args):
    random.seed(args.seed)
    with open(os.path.join(args.data_dir, task), 'r', encoding='utf-8') as json_file:
        data = json.load(json_file)
    instances = data["Instances"]
    num_available_instances = len(instances)
    instance_indices = list(range(num_available_instances))
    actual_num_test_instances = min(number_of_instances, num_available_instances)
    test_indices = random.sample(instance_indices, actual_num_test_instances)
    remaining_indices = [i for i in instance_indices if i not in test_indices]
    generic_instruction_template = ''
    current_definition = data.get("Definition", [""])[0]
    if 'Definition' in modified:
        current_definition = modified['Definition']
    if current_definition and current_definition.strip() != '-':
        generic_instruction_template = "Definition: " + current_definition.strip()
    system_message_content = ""
    test_promptlist = []
    test_answerlist = []
    for i in test_indices:
        raw_instance_input = instances[i]['input']
        instance_id = instances[i].get('id', f'held_out_test_idx_{i}')
        initial_token_ids = tokenizer.encode(raw_instance_input, add_special_tokens=False)
        original_token_length = len(initial_token_ids)
        if original_token_length <= MAX_ARTICLE_TOKENS:
            instance_input = raw_instance_input
        else:
            sentences = urdu_sent_tokenize(raw_instance_input)
            if not sentences:
                # tqdm.write(f"Warning: Sentence tokenization failed for ID {instance_id} (training_encode - test split). Using hard token truncation.")
                truncated_token_ids = initial_token_ids[:MAX_ARTICLE_TOKENS]
                instance_input = tokenizer.decode(truncated_token_ids, skip_special_tokens=True)
            else:
                selected_sentences = []
                current_total_tokens = 0
                for sentence_text in sentences:
                    if not sentence_text.strip(): continue
                    sentence_tokens_llm = tokenizer.encode(sentence_text, add_special_tokens=False)
                    if current_total_tokens + len(sentence_tokens_llm) <= MAX_ARTICLE_TOKENS:
                        selected_sentences.append(sentence_text)
                        current_total_tokens += len(sentence_tokens_llm)
                    else:
                        if not selected_sentences and len(sentence_tokens_llm) > MAX_ARTICLE_TOKENS:
                            # tqdm.write(f"Warning: First sentence ({len(sentence_tokens_llm)} tokens) for ID {instance_id} (training_encode - test split) exceeds MAX_ARTICLE_TOKENS ({MAX_ARTICLE_TOKENS}). Truncating it.")
                            truncated_first_sentence_ids = sentence_tokens_llm[:MAX_ARTICLE_TOKENS]
                            selected_sentences.append(tokenizer.decode(truncated_first_sentence_ids, skip_special_tokens=True))
                        break
                if not selected_sentences:
                    # tqdm.write(f"Warning: No sentences selected for ID {instance_id} (training_encode - test split) after truncation. Using hard token truncation.")
                    truncated_token_ids = initial_token_ids[:MAX_ARTICLE_TOKENS]
                    instance_input = tokenizer.decode(truncated_token_ids, skip_special_tokens=True)
                else:
                    instance_input = " ".join(selected_sentences)
            final_truncated_len = len(tokenizer.encode(instance_input, add_special_tokens=False))
            # if original_token_length > MAX_ARTICLE_TOKENS:
                # tqdm.write(f"ID {instance_id} (training_encode - test split): Original tokens: {original_token_length}, Truncated to: {final_truncated_len} tokens (target <={MAX_ARTICLE_TOKENS}).")
        user_content = generic_instruction_template + "\n" + "Input: " + instance_input + "\nخلاصہ:"
        test_promptlist.append([{"role": "system", "content": system_message_content}, {"role": "user", "content": user_content}])
        test_answerlist.append(instances[i]['output'][0] if isinstance(instances[i]['output'], list) else instances[i]['output'])
    train_promptlist_full, train_answerlist_full, train_indexlist_full = [], [], []
    random.seed(args.train_seed)
    actual_num_train_samples = min(args.num_train, len(remaining_indices))
    if remaining_indices and actual_num_train_samples > 0 :
        train_sample_indices_from_remaining = random.sample(remaining_indices, actual_num_train_samples)
        for i in train_sample_indices_from_remaining:
            raw_instance_input = instances[i]['input']
            instance_id = instances[i].get('id', f'train_split_idx_{i}')
            initial_token_ids = tokenizer.encode(raw_instance_input, add_special_tokens=False)
            original_token_length = len(initial_token_ids)
            if original_token_length <= MAX_ARTICLE_TOKENS:
                instance_input = raw_instance_input
            else:
                sentences = urdu_sent_tokenize(raw_instance_input)
                if not sentences:
                    # tqdm.write(f"Warning: Sentence tokenization failed for ID {instance_id} (training_encode - train split). Using hard token truncation.")
                    truncated_token_ids = initial_token_ids[:MAX_ARTICLE_TOKENS]
                    instance_input = tokenizer.decode(truncated_token_ids, skip_special_tokens=True)
                else:
                    selected_sentences = []
                    current_total_tokens = 0
                    for sentence_text in sentences:
                        if not sentence_text.strip(): continue
                        sentence_tokens_llm = tokenizer.encode(sentence_text, add_special_tokens=False)
                        if current_total_tokens + len(sentence_tokens_llm) <= MAX_ARTICLE_TOKENS:
                            selected_sentences.append(sentence_text)
                            current_total_tokens += len(sentence_tokens_llm)
                        else:
                            if not selected_sentences and len(sentence_tokens_llm) > MAX_ARTICLE_TOKENS:
                                # tqdm.write(f"Warning: First sentence ({len(sentence_tokens_llm)} tokens) for ID {instance_id} (training_encode - train split) exceeds MAX_ARTICLE_TOKENS ({MAX_ARTICLE_TOKENS}). Truncating it.")
                                truncated_first_sentence_ids = sentence_tokens_llm[:MAX_ARTICLE_TOKENS]
                                selected_sentences.append(tokenizer.decode(truncated_first_sentence_ids, skip_special_tokens=True))
                            break
                    if not selected_sentences:
                        # tqdm.write(f"Warning: No sentences selected for ID {instance_id} (training_encode - train split) after truncation. Using hard token truncation.")
                        truncated_token_ids = initial_token_ids[:MAX_ARTICLE_TOKENS]
                        instance_input = tokenizer.decode(truncated_token_ids, skip_special_tokens=True)
                    else:
                        instance_input = " ".join(selected_sentences)
                final_truncated_len = len(tokenizer.encode(instance_input, add_special_tokens=False))
                # if original_token_length > MAX_ARTICLE_TOKENS:
                    # tqdm.write(f"ID {instance_id} (training_encode - train split): Original tokens: {original_token_length}, Truncated to: {final_truncated_len} tokens (target <={MAX_ARTICLE_TOKENS}).")
            user_content = generic_instruction_template + "\n" + "Input: " + instance_input + "\nخلاصہ:"
            train_promptlist_full.append([{"role": "system", "content": system_message_content}, {"role": "user", "content": user_content}])
            train_answerlist_full.append(instances[i]['output'][0] if isinstance(instances[i]['output'], list) else instances[i]['output'])
            train_indexlist_full.append(i)
    dev_promptlist, dev_answerlist, dev_index_list = [], [], []
    return test_promptlist, test_answerlist, test_indices, train_promptlist_full, train_answerlist_full, train_indexlist_full, dev_promptlist, dev_answerlist, dev_index_list

def create_batches(instances_list, labels_list=[], batch_size=1):
    instance_batches, label_batches = [], []
    for i in range(0, len(instances_list), batch_size):
        instance_batches.append(instances_list[i:i+batch_size])
        if labels_list: label_batches.append(labels_list[i: i + batch_size])
    return (instance_batches, label_batches) if labels_list else instance_batches

def construct_urdu_instruction_prompt(mode, task_name, num_shots, num_test_instances, data_seed, null_word=None, args=args):
    if mode == "Instruction Only":
        prompt_list, answer_list, index_list = encode_urdu_instruction(
            task_name, instruction_structure=["Definition"], number_of_examples=num_shots,
            number_of_instances=num_test_instances, data_seed=data_seed, null_word=null_word, args=args)
    else: raise ValueError("Invalid mode entry, mode not recognized")
    return prompt_list, answer_list, index_list

def counter(func):
    def wrapper(*args_wrapper, **kwargs_wrapper):
        wrapper.count += 1
        global_progress_bar.update(1)
        if global_progress_bar.n >= global_progress_bar.total:
            tqdm.write("Budget exhausted based on progress bar count.")
        return func(*args_wrapper, **kwargs_wrapper)
    wrapper.count = 0
    return wrapper

@counter
def complete_phi4(prompt_messages, max_tokens=None):
    try:
        if isinstance(prompt_messages, dict): processed_prompt_messages = [prompt_messages]
        elif isinstance(prompt_messages, list) and all(isinstance(item, dict) for item in prompt_messages):
            processed_prompt_messages = prompt_messages
        else:
            tqdm.write(f"Warning: Unexpected prompt_messages format in complete_phi4: {type(prompt_messages)}")
            processed_prompt_messages = [{"role": "user", "content": str(prompt_messages)}]
        input_ids = tokenizer.apply_chat_template(processed_prompt_messages, return_tensors="pt", add_generation_prompt=True)
        # tqdm.write(f"LLM Call Input Token Count: {input_ids.shape[1]}") # Verbose
        if input_ids.shape[1] > 2000: # Reduced threshold for warning
            tqdm.write(f"WARNING: Long input to LLM: {input_ids.shape[1]} tokens. Prompt: {str(prompt_messages)[:150]}...")
    except Exception as e:
        tqdm.write(f"Error tokenizing prompt for logging in complete_phi4: {e}")
    torch.cuda.empty_cache(); gc.collect()
    args_local = generation_args.copy()
    if max_tokens is not None: args_local["max_new_tokens"] = max_tokens
    response = ""
    try:
        outputs = pipe(prompt_messages, **args_local, return_full_text=False)
        if outputs and isinstance(outputs, list) and outputs[0] and "generated_text" in outputs[0]:
            response = outputs[0]["generated_text"].strip()
        else: tqdm.write(f"Unexpected LLM output format: {outputs}")
    except RuntimeError as e:
        if "CUDA out of memory" in str(e):
            tqdm.write(f"CUDA OOM in complete_phi4. Input tokens: {input_ids.shape[1] if 'input_ids' in locals() else 'N/A'}. Max new: {args_local['max_new_tokens']}. Prompt: {str(prompt_messages)[:100]}...")
            torch.cuda.empty_cache(); gc.collect()
        else:
            tqdm.write(f"Runtime error during LLM generation: {e}")
            raise e
    except Exception as e:
        tqdm.write(f"Unexpected error during LLM generation: {e}. Input: {str(prompt_messages)[:100]}...")
    return response

def run(mode, batch_size, num_shots, chosen_task_name, num_samples, data_seed=0, override_prompts=False, function_to_call=None, split=None, modified={}, args=args):
    if not override_prompts:
        prompt_list, answer_list, index_list = construct_urdu_instruction_prompt(
            mode=mode, task_name=chosen_task_name, num_shots=num_shots,
            num_test_instances=num_samples, data_seed=data_seed, args=args)
    else:
        prompt_list, answer_list, index_list = function_to_call(
            mode=mode, task_name=chosen_task_name, num_shots=num_shots,
            num_test_instances=num_samples, data_seed=data_seed, null_word=None,
            split=split, modified=modified, args=args)
    prompt_batches, batch_test_labels = create_batches(prompt_list, answer_list, batch_size)
    predictions = []
    for batch_of_prompts, batch_of_labels in zip(prompt_batches, batch_test_labels):
        for individual_prompt_messages in batch_of_prompts:
            if complete_phi4.count >= args.budget:
                tqdm.write("Budget exhausted in run function. Stopping generation.")
                predictions.extend([""] * (len(answer_list) - len(predictions)))
                break
            pred = complete_phi4(individual_prompt_messages)
            predictions.append(pred)
        torch.cuda.empty_cache(); gc.collect()
        if complete_phi4.count >= args.budget: break
    while len(predictions) < len(answer_list): predictions.append("")
    all_rouge_scores_dicts = []
    for ref, pred_text in zip(answer_list, predictions):
        if not pred_text.strip():
            all_rouge_scores_dicts.append({"rouge1": 0.0, "rouge2": 0.0, "rougeL": 0.0})
            continue
        try: all_rouge_scores_dicts.append(calculate_urdu_rouge(ref, pred_text))
        except Exception as e: all_rouge_scores_dicts.append({"rouge1": 0.0, "rouge2": 0.0, "rougeL": 0.0})
    rouge1_scores_list = [s["rouge1"] for s in all_rouge_scores_dicts]
    rouge2_scores_list = [s["rouge2"] for s in all_rouge_scores_dicts]
    rougeL_scores_list = [s["rougeL"] for s in all_rouge_scores_dicts]
    valid_predictions = [p if p.strip() else "خالی" for p in predictions]
    valid_references = [r if r.strip() else "خالی" for r in answer_list]
    bert_f1_scores_list = [0.0] * len(answer_list)
    if valid_predictions and any(vp.strip() and vp != "خالی" for vp in valid_predictions):
        bert_device = "cpu"  # Force CPU usage as requested
        bert_model_type = "xlm-roberta-large"
        tqdm.write(f"Calculating BERTScore on device: {bert_device} with model: {bert_model_type}")
        try:
            P_bert, R_bert, F1_bert = bert_score(
                valid_predictions, valid_references, lang="ur",
                model_type=bert_model_type, device=bert_device,
                verbose=False, batch_size=8
            )
            bert_f1_scores_list = F1_bert.tolist()
            tqdm.write(f"BERTScore calculation successful on {bert_device}.")
        except Exception as e_cpu:
            tqdm.write(f"Error calculating BERT score on CPU: {e_cpu}. Defaulting to 0.")

        if not any(s > 0 for s in bert_f1_scores_list):
            tqdm.write("BERTScore resulted in all zeros, indicating a potential issue or all predictions were empty/invalid.")
    bleu_scores_list = []
    smoothie = SmoothingFunction().method4
    for ref, pred_text in zip(answer_list, predictions):
        if not pred_text.strip():
            bleu_scores_list.append(0.0)
            continue
        pred_tokens = urdu_tokenize(pred_text.lower())
        ref_tokens = urdu_tokenize(ref.lower())
        if not pred_tokens or not ref_tokens:
            bleu_scores_list.append(0.0)
            continue
        try:
            bleu_scores_list.append(sentence_bleu([ref_tokens], pred_tokens, smoothing_function=smoothie))
        except Exception as e:
            bleu_scores_list.append(0.0)
    avg_rouge1 = np.mean(rouge1_scores_list) * 100 if rouge1_scores_list else 0.0
    avg_rouge2 = np.mean(rouge2_scores_list) * 100 if rouge2_scores_list else 0.0
    avg_rougeL = np.mean(rougeL_scores_list) * 100 if rougeL_scores_list else 0.0
    avg_bert = np.mean(bert_f1_scores_list) * 100 if bert_f1_scores_list else 0.0
    avg_bleu = np.mean(bleu_scores_list) * 100 if bleu_scores_list else 0.0
    return (predictions, avg_rouge1, avg_rouge2, avg_rougeL, avg_bert, avg_bleu,
            answer_list, index_list, rouge1_scores_list, rouge2_scores_list,
            rougeL_scores_list, bert_f1_scores_list, bleu_scores_list)


# Initial Setup
meta_path = os.path.join(args.meta_dir, args.meta_name)
# Open meta_file in 'w+' mode at the beginning to truncate/create it.
# It will be opened in 'a' mode within the loop for appending.
meta_file_initial_open = open(meta_path, 'w+', encoding='utf-8')
# We don't write to it immediately here, but it ensures the file is ready.
# It will be closed at the very end of the script.

batch_size = args.batch_size
num_shots = args.num_shots
mode = args.mode
seed = args.seed
train_seed = args.train_seed
urdu_task_files = [f for f in os.listdir(args.data_dir) if f.endswith('.json')]
assert args.task_idx >= 0 and args.task_idx < len(urdu_task_files), "Invalid task index"
chosen_task_name = urdu_task_files[args.task_idx]
tqdm.write("Running Experiment for: " + chosen_task_name)
if 'wandb' in globals() and wandb.run:
    wandb.log({"Experiment": f"Running Experiment for: {chosen_task_name} with Llama-3.1-8B-Instruct", "Model": model_name})
nltk.download('punkt', quiet=True)
num_samples = 100 # Number of samples for final testing
num_train_samples = args.num_train # Number of samples for GA evaluation
np.random.seed(args.train_seed); torch.manual_seed(args.train_seed)

# add urdu prompt instruction
instruction = ""
if args.agnostic:
    instruction = "آپ کو ایک متن دیا گیا ہے۔ اسے احتیاط سے پڑھیں اور سمجھیں، اور ایک مختصر خلاصہ فراہم کریں۔"
if 'wandb' in globals() and wandb.run:
    wandb.log({"Num Compose":args.num_compose,"Num Candidates":args.num_candidates,"Max Iterations":args.num_iter,
               "Tournament Selections":args.tournament_selection,"Edit Operations":args.edits,
               "Population Size":args.population_size,"Num Offspring":args.num_offspring,"Patience":args.patience,
               "Mutation Probability":args.mutation_prob})

def evaluate_objectives(candidate_prompt, split="train"):
    if candidate_prompt in evaluation_cache and split == evaluation_cache[candidate_prompt].get("split"):
        cached_data = evaluation_cache[candidate_prompt]
        return (cached_data["summarization_score"], cached_data["grammar_score"], cached_data["avg_rouge1"],
                cached_data["avg_rouge2"], cached_data["avg_rougeL"], cached_data["avg_bert"], cached_data["avg_bleu"])
    if complete_phi4.count >= args.budget:
        tqdm.write("Budget exhausted in evaluate_objectives. Returning 0 scores.")
        return 0, 0, 0, 0, 0, 0, 0
    (predictions, avg_r1, avg_r2, avg_rL, avg_bert, avg_bleu, _, _, _, _, _, _, _) = run(
        mode=args.mode, batch_size=args.batch_size, num_shots=args.num_shots,
        chosen_task_name=chosen_task_name,
        num_samples=num_train_samples if split == "train" else num_samples,
        data_seed=args.seed if split == "test" else args.train_seed,
        override_prompts=True, function_to_call=custom_urdu_instruction_prompt,
        split=split, modified={"Definition": candidate_prompt}, args=args)
    summarization_score = (0.6 * avg_rL + 0.4 * avg_bert)
    grammar_score = get_urdu_grammar_score(candidate_prompt)
    with open(os.path.join(args.meta_dir, "rouge_scores.txt"), "a", encoding="utf-8") as f_rouge:
        f_rouge.write(f"Candidate: {candidate_prompt}\nSplit: {split}\nAvg R1: {avg_r1:.4f}, Avg R2: {avg_r2:.4f}, Avg RL: {avg_rL:.4f}\n\n")
    with open(os.path.join(args.meta_dir, "bert_scores.txt"), "a", encoding="utf-8") as f_bert:
        f_bert.write(f"Candidate: {candidate_prompt}\nSplit: {split}\nAvg BERT: {avg_bert:.4f}\n\n")
    with open(os.path.join(args.meta_dir, "bleu_scores.txt"), "a", encoding="utf-8") as f_bleu:
        f_bleu.write(f"Candidate: {candidate_prompt}\nSplit: {split}\nAvg BLEU: {avg_bleu:.4f}\n\n")
    evaluation_cache[candidate_prompt] = {
        "summarization_score": summarization_score, "grammar_score": grammar_score,
        "avg_rouge1": avg_r1, "avg_rouge2": avg_r2, "avg_rougeL": avg_rL,
        "avg_bert": avg_bert, "avg_bleu": avg_bleu, "split": split}
    return summarization_score, grammar_score, avg_r1, avg_r2, avg_rL, avg_bert, avg_bleu

def score_final(candidate_prompt, split="test", write=False):
    (predictions, avg_r1, avg_r2, avg_rL, avg_bert, avg_bleu, answer_list, indexlist,
     r1_scores_list, r2_scores_list, rL_scores_list, bert_scores_list, bleu_scores_list) = run(
        mode=args.mode, batch_size=args.batch_size, num_shots=args.num_shots,
        chosen_task_name=chosen_task_name, num_samples=num_samples, data_seed=args.seed,
        override_prompts=True, function_to_call=custom_urdu_instruction_prompt,
        split=split, modified={"Definition": candidate_prompt}, args=args)
    summarization_score = 0.6 * avg_rL + 0.4 * avg_bert
    if split == "test" and write:
        pname = args.meta_name.split(".")[0] + "_predictions.json"
        detailed_predictions_output = []
        for i in range(len(predictions)):
            detailed_predictions_output.append({
                "id": indexlist[i] if i < len(indexlist) else None,
                "reference_summary": answer_list[i] if i < len(answer_list) else None,
                "generated_summary": predictions[i] if i < len(predictions) else None,
                "rouge1": r1_scores_list[i] if i < len(r1_scores_list) else 0.0,
                "rouge2": r2_scores_list[i] if i < len(r2_scores_list) else 0.0,
                "rougeL": rL_scores_list[i] if i < len(rL_scores_list) else 0.0,
                "bert_score": bert_scores_list[i] if i < len(bert_scores_list) else 0.0,
                "bleu_score": bleu_scores_list[i] if i < len(bleu_scores_list) else 0.0,})
        pred_dump = {"final_prompt": candidate_prompt, "overall_avg_rouge1": avg_r1,
                     "overall_avg_rouge2": avg_r2, "overall_avg_rougeL": avg_rL,
                     "overall_avg_bert": avg_bert, "overall_avg_bleu": avg_bleu,
                     "predictions_detailed": detailed_predictions_output}
        ppath = os.path.join(args.meta_dir, pname)
        with open(ppath, "w+", encoding="utf-8") as pfile:
            json.dump(pred_dump, pfile, ensure_ascii=False, indent=2)
    return summarization_score, avg_r1, avg_r2, avg_rL, avg_bert, avg_bleu

def custom_urdu_instruction_prompt(mode=args.mode, task_name=None, num_shots=args.num_shots, num_test_instances=100, data_seed=None, null_word=None, split='train', modified={}, args=args):
    if task_name is None: task_name = chosen_task_name
    if mode == "Instruction Only":
        result = training_encode_urdu_instruction(
            task_name, instruction_structure=["Definition"], number_of_examples=num_shots,
            number_of_instances=num_test_instances, data_seed=data_seed, null_word=null_word,
            modified=modified, args=args)
    else: raise ValueError("Invalid mode for custom_urdu_instruction_prompt")
    if split == 'test': return result[0], result[1], result[2]
    elif split == 'train': return result[3], result[4], result[5]
    else: raise ValueError(f"Invalid split '{split}' for custom_urdu_instruction_prompt")

def tournament_selection(population, population_scores_tuples, num_tournaments):
    parent_indices = random.sample(range(len(population)), k=num_tournaments)
    best_parent_idx = -1
    best_parent_combined_score = -float('inf')
    for idx in parent_indices:
        combined_score = WEIGHT_SUMM * population_scores_tuples[idx][0] + WEIGHT_GRAM * population_scores_tuples[idx][1]
        if combined_score > best_parent_combined_score:
            best_parent_combined_score = combined_score
            best_parent_idx = idx
    return population[best_parent_idx], population_scores_tuples[best_parent_idx]

def urdu_crossover(parent_1_text, parent_2_text):
    try:
        phrases_1 = get_urdu_phrases(parent_1_text)
        phrases_2 = get_urdu_phrases(parent_2_text)
    except Exception: return parent_1_text, True
    if not phrases_1 or not phrases_2: return random.choice([parent_1_text, parent_2_text]), True
    p1_cross_point = random.randint(0, len(phrases_1))
    p2_cross_point = random.randint(0, len(phrases_2))
    offspring_phrases_part1 = phrases_1[:p1_cross_point] + phrases_2[p2_cross_point:]
    offspring_phrases_part2 = phrases_2[:p2_cross_point] + phrases_1[p1_cross_point:]
    chosen_offspring_phrases = random.choice([offspring_phrases_part1, offspring_phrases_part2])
    if not chosen_offspring_phrases: return random.choice([parent_1_text, parent_2_text]), True
    offspring_text = " ".join(chosen_offspring_phrases)
    offspring_text = ' '.join(urdu_tokenize(offspring_text))
    grammar_score = get_urdu_grammar_score(offspring_text)
    if is_urdu_text(offspring_text) and grammar_score >= 30: return offspring_text, False
    else: return random.choice([parent_1_text, parent_2_text]), True

def plot_pareto_front(population_data_tuples, fronts_indices, iteration, meta_dir_path, final=False):
    plt.figure(figsize=(10, 8))
    all_summ_scores = [data[1] for data in population_data_tuples]
    all_gram_scores = [data[2] for data in population_data_tuples]
    plt.scatter(all_summ_scores, all_gram_scores, c='lightgray', alpha=0.6, label='All Population Candidates', s=50)
    colors = ['red', 'blue', 'green', 'purple', 'orange', 'brown', 'pink', 'olive', 'cyan']
    for front_idx, front_candidate_indices in enumerate(fronts_indices):
        if not front_candidate_indices: continue
        color = colors[front_idx % len(colors)]
        front_summ = [population_data_tuples[i][1] for i in front_candidate_indices]
        front_gram = [population_data_tuples[i][2] for i in front_candidate_indices]
        sorted_indices_within_front = np.argsort(front_summ)
        front_summ_sorted = np.array(front_summ)[sorted_indices_within_front]
        front_gram_sorted = np.array(front_gram)[sorted_indices_within_front]
        label = f'Pareto Front {front_idx + 1}' if front_idx > 0 else 'Pareto Optimal Front (F1)'
        plt.scatter(front_summ, front_gram, c=color, label=label, s=70, edgecolors='black')
        if len(front_summ_sorted) > 1 : plt.plot(front_summ_sorted, front_gram_sorted, c=color, linestyle='--', marker='o', markersize=5)
    plt.xlabel('Summarization Score (Higher is Better)'); plt.ylabel('Grammar Score (Higher is Better)')
    title_str = f'Pareto Optimal Fronts (Final)' if final else f'Pareto Optimal Fronts (Iteration {iteration+1})'
    plt.title(title_str, fontsize=16); plt.legend(loc='best'); plt.grid(True, linestyle=':', alpha=0.7); plt.tight_layout()
    plot_name = 'pareto_final.png' if final else f'pareto_iteration_{iteration+1}.png'
    plot_path = os.path.join(meta_dir_path, plot_name)
    plt.savefig(plot_path); plt.close()
    if 'wandb' in globals() and wandb.run:
        try: wandb.log({title_str: wandb.Image(plot_path)}, step=iteration if not final else args.num_iter +1) # Log final plot at a later step
        except Exception as e: tqdm.write(f"WandB logging error for Pareto plot: {e}")
    return plot_path

WEIGHT_SUMM = 0.6
WEIGHT_GRAM = 0.4
BEST_CANDIDATE_WEIGHT_SUMM = 0.97
BEST_CANDIDATE_WEIGHT_GRAM = 0.03

def non_dominated_sorting(population_data_list):
    n = len(population_data_list)
    if n == 0: return []
    domination_count = [0] * n
    dominated_solutions = [[] for _ in range(n)]
    fronts = [[]]
    for i in range(n):
        for j in range(i + 1, n):
            p_summ, p_gram = population_data_list[i][1], population_data_list[i][2]
            q_summ, q_gram = population_data_list[j][1], population_data_list[j][2]
            p_dominates_q = (p_summ >= q_summ and p_gram >= q_gram) and (p_summ > q_summ or p_gram > q_gram)
            q_dominates_p = (q_summ >= p_summ and q_gram >= p_gram) and (q_summ > p_summ or q_gram > q_gram)
            if p_dominates_q:
                dominated_solutions[i].append(j); domination_count[j] += 1
            elif q_dominates_p:
                dominated_solutions[j].append(i); domination_count[i] += 1
    for i in range(n):
        if domination_count[i] == 0: fronts[0].append(i)
    front_idx = 0
    while fronts[front_idx]:
        next_front_indices = []
        for i in fronts[front_idx]:
            for j in dominated_solutions[i]:
                domination_count[j] -= 1
                if domination_count[j] == 0: next_front_indices.append(j)
        front_idx += 1
        if next_front_indices: fronts.append(next_front_indices)
        else: break
    return fronts

def compute_crowding_distance(population_data_list, front_indices):
    if not front_indices: return {}
    num_in_front = len(front_indices)
    distances = {idx: 0.0 for idx in front_indices}
    num_objectives = 2
    for m in range(num_objectives):
        objective_idx_in_tuple = m + 1
        sorted_front_by_obj_m = sorted(front_indices, key=lambda i: population_data_list[i][objective_idx_in_tuple])
        distances[sorted_front_by_obj_m[0]] = float('inf')
        if num_in_front > 1: distances[sorted_front_by_obj_m[-1]] = float('inf')
        range_obj_m = 0
        if num_in_front > 1:
            obj_m_min = population_data_list[sorted_front_by_obj_m[0]][objective_idx_in_tuple]
            obj_m_max = population_data_list[sorted_front_by_obj_m[-1]][objective_idx_in_tuple]
            range_obj_m = obj_m_max - obj_m_min
        if num_in_front > 2 and range_obj_m > 1e-6:
            for k in range(1, num_in_front - 1):
                idx_k = sorted_front_by_obj_m[k]
                idx_k_plus_1 = sorted_front_by_obj_m[k+1]
                idx_k_minus_1 = sorted_front_by_obj_m[k-1]
                distances[idx_k] += (population_data_list[idx_k_plus_1][objective_idx_in_tuple] -
                                     population_data_list[idx_k_minus_1][objective_idx_in_tuple]) / range_obj_m
    return distances

def plot_best_candidate_scores(meta_dir_path, r1_scores, r2_scores, rL_scores, b_scores, bl_scores, summ_scores, gram_scores, comb_scores):
    iterations = list(range(len(rL_scores)))
    score_types = {"ROUGE-1": r1_scores, "ROUGE-2": r2_scores, "ROUGE-L": rL_scores,
                   "BERT": b_scores, "BLEU": bl_scores, "Summarization": summ_scores,
                   "Grammar": gram_scores, "Combined (0.97S+0.03G)": comb_scores}
    for score_name, scores_data in score_types.items():
        plt.figure(figsize=(10, 6))
        plt.plot(iterations, scores_data, marker='o', linestyle='-', markersize=5, label=f'Best {score_name}')
        plt.xlabel('Iteration Number (0 = Initial)'); plt.ylabel(f'{score_name} Score')
        plt.title(f'Best Candidate {score_name} Score vs. Iteration', fontsize=14)
        plt.grid(True, linestyle=':', alpha=0.7); plt.xticks(iterations); plt.legend(); plt.tight_layout()
        plot_path = os.path.join(meta_dir_path, f'history_best_{score_name.lower().replace(" ", "_").replace("-", "_").replace("(", "").replace(")", "").replace(".","").replace("+","")}_scores.png')
        plt.savefig(plot_path); plt.close()
        if 'wandb' in globals() and wandb.run:
            try: wandb.log({f"History Best {score_name} Scores Plot": wandb.Image(plot_path)}, step=iterations[-1] if iterations else 0)
            except Exception as e: tqdm.write(f"WandB logging error for score history plot {score_name}: {e}")


def adaptive_mutation_probability(iteration, max_iterations, base_prob=0.5):
    """Adaptive mutation probability that decreases over time"""
    # Start high, decrease as we progress
    decay_factor = 1 - (iteration / max_iterations) * 0.3
    return base_prob * decay_factor

# --- Main Evolutionary Loop ---
W_candidates = [instruction] * args.population_size
W_deletesets = [[] for _ in range(args.population_size)]

with open(meta_path, 'a', encoding='utf-8') as meta_append_initial:
    meta_append_initial.write(f"Initial Urdu Candidate:\t {instruction}\n")
    tqdm.write(f"Initial Urdu Candidate: {instruction}")
    torch.cuda.empty_cache(); gc.collect()
    # Evaluate initial instruction
    s_score, g_score, r1_score, r2_score, rL_score, b_score, bl_score = evaluate_objectives(instruction, split='train')
    W_objectives = [(s_score, g_score)] * args.population_size # Initialize objectives for the whole population
    initial_scores_tuple = (r1_score, r2_score, rL_score, b_score, bl_score, s_score, g_score)
    W_all_scores = [initial_scores_tuple] * args.population_size # Store all scores for the population

    meta_append_initial.write(
        "Initial Objectives (Summarization, Grammar, ROUGE-1, ROUGE-2, ROUGE-L, BERT, BLEU):\t "
        f"({s_score:.2f}, {g_score:.2f}, {r1_score:.2f}, {r2_score:.2f}, {rL_score:.2f}, {b_score:.2f}, {bl_score:.2f})\n")
    tqdm.write(
        "Initial Objectives (Summarization, Grammar, ROUGE-1, ROUGE-2, ROUGE-L, BERT, BLEU): "
        f"({s_score:.2f}, {g_score:.2f}, {r1_score:.2f}, {r2_score:.2f}, {rL_score:.2f}, {b_score:.2f}, {bl_score:.2f})")

if 'wandb' in globals() and wandb.run:
    wandb.log({"Initial Urdu Candidate": instruction, "Initial Summarization Score": s_score, "Initial Grammar Score": g_score,
               "Initial ROUGE-1 Score": r1_score, "Initial ROUGE-2 Score": r2_score, "Initial ROUGE-L Score": rL_score,
               "Initial BERT Score": b_score, "Initial BLEU Score": bl_score, "Iteration": 0})

history_best_rouge1 = [r1_score]; history_best_rouge2 = [r2_score]; history_best_rougeL = [rL_score]
history_best_bert = [b_score]; history_best_bleu = [bl_score]
history_best_summarization = [s_score]; history_best_grammar = [g_score]
history_best_combined = [BEST_CANDIDATE_WEIGHT_SUMM * s_score + BEST_CANDIDATE_WEIGHT_GRAM * g_score]
overall_best_candidate_text = instruction
overall_best_candidate_objectives = (s_score, g_score)
overall_best_combined_score = BEST_CANDIDATE_WEIGHT_SUMM * s_score + BEST_CANDIDATE_WEIGHT_GRAM * g_score
patience_counter = 0; start_time = time.time(); iter_idx = 0 # Initialize iter_idx

for iter_idx in range(args.num_iter):
    if complete_phi4.count >= args.budget:
        tqdm.write("Budget exhausted before starting iteration. Stopping.")
        break
    with open(meta_path, 'a', encoding='utf-8') as current_iter_meta_file:
        current_iter_meta_file.write(f"\n----- Iteration: {iter_idx + 1} -----\n")
        tqdm.write(f"\n----- Iteration: {iter_idx + 1} -----")
        current_iter_meta_file.write("Population at START of Iteration:\n"); tqdm.write("Population at START of Iteration:")
        population_log_start = []
        for i_pop, cand_text in enumerate(W_candidates):
            # Ensure W_all_scores has valid tuples
            current_scores = W_all_scores[i_pop] if i_pop < len(W_all_scores) and isinstance(W_all_scores[i_pop], tuple) and len(W_all_scores[i_pop]) == 7 else (0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0)
            r1, r2, rL, bert, bleu, s, g = current_scores
            pop_entry = {"prompt": cand_text, "summ_score": s, "gram_score": g, "r1": r1, "r2": r2, "rL": rL, "bert": bert, "bleu": bleu}
            population_log_start.append(pop_entry)
            log_line = (f"  Cand {i_pop+1}: Summ={s:.2f}, Gram={g:.2f}, R1={r1:.2f}, R2={r2:.2f}, RL={rL:.2f}, BERT={bert:.2f}, BLEU={bleu:.2f}"
                        f" - '{cand_text[:50]}...'\n")
            current_iter_meta_file.write(log_line); tqdm.write(log_line.strip())
        if 'wandb' in globals() and wandb.run: wandb.log({f"Population Start (Iter {iter_idx+1})": population_log_start}, step=iter_idx+1)

        offspring_candidates, offspring_deletesets = [], []
        if args.num_offspring > 0:
            for j_offspring in range(args.num_offspring // 2):
                if complete_phi4.count >= args.budget: break
                parent1_text, _ = tournament_selection(W_candidates, W_objectives, args.tournament_selection)
                parent2_text, _ = tournament_selection(W_candidates, W_objectives, args.tournament_selection)
                if parent1_text == parent2_text and len(W_candidates) > 1:
                    temp_population_for_tournament = [c for c in W_candidates if c != parent1_text]
                    if temp_population_for_tournament:
                        temp_objectives_indices = [i for i, c in enumerate(W_candidates) if c != parent1_text]
                        temp_objectives_for_tournament = [W_objectives[idx] for idx in temp_objectives_indices]
                        if temp_objectives_for_tournament: # Ensure not empty
                           parent2_text, _ = tournament_selection(temp_population_for_tournament, temp_objectives_for_tournament, args.tournament_selection)
                current_iter_meta_file.write(f"  Parent 1 (Iter {iter_idx+1}, OffspringSet {j_offspring+1}):\t {parent1_text}\n")
                current_iter_meta_file.write(f"  Parent 2 (Iter {iter_idx+1}, OffspringSet {j_offspring+1}):\t {parent2_text}\n")
                child1_text, err1 = urdu_crossover(parent1_text, parent2_text) # Ensure urdu_crossover is defined
                child2_text, err2 = urdu_crossover(parent2_text, parent1_text)
                if not err1 and child1_text not in offspring_candidates + W_candidates:
                    offspring_candidates.append(child1_text)
                    offspring_deletesets.append(list(W_deletesets[W_candidates.index(parent1_text)]) if parent1_text in W_candidates else [])
                if not err2 and child2_text not in offspring_candidates + W_candidates:
                    offspring_candidates.append(child2_text)
                    offspring_deletesets.append(list(W_deletesets[W_candidates.index(parent2_text)]) if parent2_text in W_candidates else [])

        mutated_candidates_texts, mutated_deletesets = [], []
        candidates_for_mutation = list(W_candidates)
        # deletesets_for_mutation = [list(ds) for ds in W_deletesets] # This was from original, ensure W_deletesets is aligned

        for i_mut in range(len(candidates_for_mutation)):
            if complete_phi4.count >= args.budget: break

            # Determine mutation probability (fixed or adaptive)
            current_mutation_prob = args.mutation_prob
            # if callable(globals().get('adaptive_mutation_probability')):
            #    current_mutation_prob = adaptive_mutation_probability(iter_idx, args.num_iter, args.mutation_prob)


            if random.random() < current_mutation_prob:
                base_candidate_text = candidates_for_mutation[i_mut]
                # Get the corresponding deleteset for the base_candidate_text
                try:
                    original_idx_for_mutation = W_candidates.index(base_candidate_text)
                    base_deleteset = list(W_deletesets[original_idx_for_mutation])
                except ValueError: # Fallback if not found (should ideally not happen)
                    base_deleteset = []

                current_iter_meta_file.write(f"  Mutating Candidate {i_mut+1}: {base_candidate_text}\n")
                tqdm.write(f"  Mutating Candidate {i_mut+1}: {base_candidate_text[:70]}...")

                mutated_text_current = base_candidate_text
                current_deleteset_for_mutation = list(base_deleteset)

                # Simplified/Placeholder for your actual mutation logic (simple_urdu_mutation or complex)
                if len(urdu_tokenize(base_candidate_text)) <= 8 and callable(globals().get('simple_urdu_mutation')):
                     mutated_text_current = simple_urdu_mutation(base_candidate_text, meta_file_handle=current_iter_meta_file)
                     # simple_urdu_mutation might not return deleteset, so current_deleteset_for_mutation remains base_deleteset
                else: # Use the more complex mutation
                    try:
                        phrase_list_for_mutation = get_urdu_phrases(base_candidate_text)
                        current_iter_meta_file.write(f"    Extracted Phrases for Mutation: {phrase_list_for_mutation}\n")
                        tqdm.write(f"    Extracted Phrases: {phrase_list_for_mutation}")
                    except Exception as e:
                        current_iter_meta_file.write(f"    Error getting phrases for mutation: {e}\n"); tqdm.write(f"    Error getting phrases: {e}")
                        continue

                    for edit_idx_compose in range(args.num_compose):
                        if not args.edits: continue
                        available_edits = list(args.edits)
                        if not phrase_list_for_mutation or len(phrase_list_for_mutation) < 2:
                            if 'swap' in available_edits: available_edits.remove('swap')
                        if not phrase_list_for_mutation:
                            if 'del' in available_edits: available_edits.remove('del')
                            if 'sub' in available_edits: available_edits.remove('sub')
                        if not current_deleteset_for_mutation and 'add' in available_edits: available_edits.remove('add')
                        if not available_edits:
                            current_iter_meta_file.write(f"    ComposeStep {edit_idx_compose+1}: No available edit operations.\n")
                            tqdm.write(f"    ComposeStep {edit_idx_compose+1}: No available edit operations.")
                            break
                        chosen_edit_op = random.choice(available_edits)
                        mutated_text_current, current_deleteset_for_mutation = safe_urdu_mutation(
                            chosen_edit_op, mutated_text_current, list(phrase_list_for_mutation),
                            current_deleteset_for_mutation, meta_file_handle=current_iter_meta_file,
                            mutation_step_info=f"Cand {i_mut+1} ComposeStep {edit_idx_compose+1}")
                        if mutated_text_current != base_candidate_text and edit_idx_compose < args.num_compose - 1 :
                            try: phrase_list_for_mutation = get_urdu_phrases(mutated_text_current)
                            except Exception: phrase_list_for_mutation = []

                if mutated_text_current != base_candidate_text and mutated_text_current not in mutated_candidates_texts + W_candidates + offspring_candidates:
                    mutated_candidates_texts.append(mutated_text_current)
                    mutated_deletesets.append(current_deleteset_for_mutation)

        # Combine parent, offspring, and mutated candidates
        combined_candidates_texts_pool = W_candidates + offspring_candidates + mutated_candidates_texts

        # Create a mapping for deletesets to handle unique candidates correctly
        deletesets_pool_map = {}
        for i, txt in enumerate(W_candidates): deletesets_pool_map.setdefault(txt, W_deletesets[i])
        for i, txt in enumerate(offspring_candidates): deletesets_pool_map.setdefault(txt, offspring_deletesets[i])
        for i, txt in enumerate(mutated_candidates_texts): deletesets_pool_map.setdefault(txt, mutated_deletesets[i])

        unique_combined_texts_list = []
        seen_texts_for_eval = set()
        for txt_comb in combined_candidates_texts_pool:
            if txt_comb not in seen_texts_for_eval:
                unique_combined_texts_list.append(txt_comb)
                seen_texts_for_eval.add(txt_comb)

        final_combined_deletesets_for_eval = [deletesets_pool_map.get(text, []) for text in unique_combined_texts_list]

        tqdm.write(f"Evaluating {len(unique_combined_texts_list)} unique candidates in combined pool for Iter {iter_idx + 1}")
        evaluated_objectives_list = [] # Stores (summ, gram)
        all_candidate_scores_for_iter_log = [] # Stores (r1,r2,rL,B,BL,S,G) for logging

        for i_eval, cand_text_eval in enumerate(unique_combined_texts_list):
            if complete_phi4.count >= args.budget:
                num_remaining_to_eval = len(unique_combined_texts_list) - len(evaluated_objectives_list)
                evaluated_objectives_list.extend([(-1.0, -1.0)] * num_remaining_to_eval)
                all_candidate_scores_for_iter_log.extend([(0,0,0,0,0,-1.0,-1.0)] * num_remaining_to_eval)
                break
            s, g, r1, r2, rL, bert, bleu = evaluate_objectives(cand_text_eval, split='train')
            evaluated_objectives_list.append((s, g))
            all_candidate_scores_for_iter_log.append((r1, r2, rL, bert, bleu, s, g))

        # Prepare data for non-dominated sorting: list of (text, summ_score, gram_score, deleteset)
        current_population_data_for_sorting = []
        for i_data in range(len(unique_combined_texts_list)):
            summ_s, gram_s = evaluated_objectives_list[i_data]
            deleteset_val = final_combined_deletesets_for_eval[i_data]
            current_population_data_for_sorting.append(
                (unique_combined_texts_list[i_data], summ_s, gram_s, deleteset_val)
            )

        # --- Non-Dominated Sorting and Selection (Restored from original logic) ---
        fronts = non_dominated_sorting(current_population_data_for_sorting)
        if fronts and fronts[0] and callable(globals().get('plot_pareto_front')):
            plot_pareto_front(current_population_data_for_sorting, fronts, iter_idx, args.meta_dir)

        next_gen_indices = [] # Stores indices from current_population_data_for_sorting
        remaining_population_slots = args.population_size

        for front_indices_in_data in fronts:
            if not front_indices_in_data: continue
            if remaining_population_slots == 0: break

            if len(front_indices_in_data) <= remaining_population_slots:
                next_gen_indices.extend(front_indices_in_data)
                remaining_population_slots -= len(front_indices_in_data)
            else:
                if callable(globals().get('compute_crowding_distance')):
                    # compute_crowding_distance expects population_data_list and indices for that list
                    distances = compute_crowding_distance(current_population_data_for_sorting, front_indices_in_data)
                    sorted_front_by_crowding = sorted(front_indices_in_data, key=lambda i_crowd: distances.get(i_crowd, 0.0), reverse=True)
                    next_gen_indices.extend(sorted_front_by_crowding[:remaining_population_slots])
                else:
                    tqdm.write("Warning: compute_crowding_distance not found. Using simple truncation.")
                    next_gen_indices.extend(front_indices_in_data[:remaining_population_slots])
                remaining_population_slots = 0

        # Populate temporary lists for the selected generation
        temp_W_candidates = [current_population_data_for_sorting[i][0] for i in next_gen_indices]
        temp_W_objectives = [(current_population_data_for_sorting[i][1], current_population_data_for_sorting[i][2]) for i in next_gen_indices]
        temp_W_deletesets = [current_population_data_for_sorting[i][3] for i in next_gen_indices]
        temp_W_all_scores = [all_candidate_scores_for_iter_log[i] for i in next_gen_indices]

        # --- Padding to ensure full population size ---
        final_W_candidates = []
        final_W_objectives = []
        final_W_deletesets = []
        final_W_all_scores = []

        num_actually_selected = len(temp_W_candidates)

        if num_actually_selected >= args.population_size:
            final_W_candidates = temp_W_candidates[:args.population_size]
            final_W_objectives = temp_W_objectives[:args.population_size]
            final_W_deletesets = temp_W_deletesets[:args.population_size]
            final_W_all_scores = temp_W_all_scores[:args.population_size]
        else:
            final_W_candidates.extend(temp_W_candidates)
            final_W_objectives.extend(temp_W_objectives)
            final_W_deletesets.extend(temp_W_deletesets)
            final_W_all_scores.extend(temp_W_all_scores)

            padding_needed = args.population_size - num_actually_selected
            if num_actually_selected > 0:
                # Pad by duplicating the best individuals from the selected set (temp_W_*)
                # Sort temp_W_* by combined score to pick "best" for padding
                padding_source_with_scores = []
                for i_pad_src in range(num_actually_selected):
                    s_pad, g_pad = temp_W_objectives[i_pad_src]
                    # Handle non-numeric scores if they occur
                    s_pad_num = s_pad if isinstance(s_pad, (int, float)) else 0.0
                    g_pad_num = g_pad if isinstance(g_pad, (int, float)) else 0.0
                    combined_score_pad = WEIGHT_SUMM * s_pad_num + WEIGHT_GRAM * g_pad_num
                    padding_source_with_scores.append(
                        (temp_W_candidates[i_pad_src], temp_W_objectives[i_pad_src], temp_W_deletesets[i_pad_src], combined_score_pad, temp_W_all_scores[i_pad_src])
                    )
                padding_source_with_scores.sort(key=lambda x_pad: x_pad[3], reverse=True)

                for i_pad in range(padding_needed):
                    source_idx_pad = i_pad % len(padding_source_with_scores)
                    final_W_candidates.append(padding_source_with_scores[source_idx_pad][0])
                    final_W_objectives.append(padding_source_with_scores[source_idx_pad][1])
                    final_W_deletesets.append(padding_source_with_scores[source_idx_pad][2])
                    final_W_all_scores.append(padding_source_with_scores[source_idx_pad][4])
            else:
                tqdm.write(f"Warning: No candidates selected by Pareto. Padding with initial instruction.")
                # s_score, g_score are from the very beginning (initial instruction eval)
                initial_obj_tuple_for_pad = (s_score if 's_score' in locals() else 0.0, g_score if 'g_score' in locals() else 0.0)
                for _ in range(padding_needed):
                    final_W_candidates.append(instruction)
                    final_W_objectives.append(initial_obj_tuple_for_pad)
                    final_W_deletesets.append([])
                    final_W_all_scores.append(initial_scores_tuple)

        # Update main population lists
        W_candidates = final_W_candidates
        W_objectives = final_W_objectives
        W_deletesets = final_W_deletesets
        W_all_scores = final_W_all_scores
        # --- End of Non-Dominated Sorting, Selection, and Padding ---

        current_iter_meta_file.write("Population at END of Iteration (Selected for Next Gen):\n"); tqdm.write("Population at END of Iteration (Selected for Next Gen):")
        population_log_end = []
        for i_pop_end, cand_text_end in enumerate(W_candidates):
            current_scores_end = W_all_scores[i_pop_end] if i_pop_end < len(W_all_scores) and isinstance(W_all_scores[i_pop_end], tuple) and len(W_all_scores[i_pop_end]) == 7 else (0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0)
            r1_end, r2_end, rL_end, bert_end, bleu_end, s_end, g_end = current_scores_end
            pop_entry_end = {"prompt": cand_text_end, "summ_score": s_end, "gram_score": g_end, "r1": r1_end, "r2": r2_end, "rL": rL_end, "bert": bert_end, "bleu": bleu_end}
            population_log_end.append(pop_entry_end)
            log_line_end = (f"  Cand {i_pop_end+1}: Summ={s_end:.2f}, Gram={g_end:.2f}, R1={r1_end:.2f}, R2={r2_end:.2f}, RL={rL_end:.2f}, BERT={bert_end:.2f}, BLEU={bleu_end:.2f}"
                            f" - '{cand_text_end[:50]}...'\n")
            current_iter_meta_file.write(log_line_end); tqdm.write(log_line_end.strip())
        if 'wandb' in globals() and wandb.run: wandb.log({f"Population End (Iter {iter_idx+1})": population_log_end}, step=iter_idx+1)

        # Determine best candidate of the iteration for logging and patience (from the first Pareto front)
        iter_best_candidate_text, iter_best_objectives, iter_best_scores_tuple = "N/A", (-1.0,-1.0), (0,0,0,0,0,-1.0,-1.0)
        if fronts and fronts[0]:
            pareto_front_indices_for_log = fronts[0] # Indices for current_population_data_for_sorting
            best_idx_in_pareto_for_log = -1
            current_max_weighted_score_in_pareto_for_log = -float('inf')

            for idx_in_data_for_log in pareto_front_indices_for_log:
                s_val_log, g_val_log = current_population_data_for_sorting[idx_in_data_for_log][1], current_population_data_for_sorting[idx_in_data_for_log][2]
                s_val_log_num = s_val_log if isinstance(s_val_log, (int, float)) else 0.0
                g_val_log_num = g_val_log if isinstance(g_val_log, (int, float)) else 0.0
                weighted_score_log = BEST_CANDIDATE_WEIGHT_SUMM * s_val_log_num + BEST_CANDIDATE_WEIGHT_GRAM * g_val_log_num
                if weighted_score_log > current_max_weighted_score_in_pareto_for_log:
                    current_max_weighted_score_in_pareto_for_log = weighted_score_log
                    best_idx_in_pareto_for_log = idx_in_data_for_log

            if best_idx_in_pareto_for_log != -1:
                iter_best_candidate_text = current_population_data_for_sorting[best_idx_in_pareto_for_log][0]
                iter_best_objectives = (current_population_data_for_sorting[best_idx_in_pareto_for_log][1], current_population_data_for_sorting[best_idx_in_pareto_for_log][2])

                # The index best_idx_in_pareto_for_log is for current_population_data_for_sorting,
                # which is aligned with unique_combined_texts_list and all_candidate_scores_for_iter_log
                if best_idx_in_pareto_for_log < len(all_candidate_scores_for_iter_log):
                     iter_best_scores_tuple = all_candidate_scores_for_iter_log[best_idx_in_pareto_for_log]
                else: # Fallback if something is misaligned
                     iter_best_scores_tuple = (0,0,0,0,0, iter_best_objectives[0], iter_best_objectives[1])

        history_best_rouge1.append(iter_best_scores_tuple[0]); history_best_rouge2.append(iter_best_scores_tuple[1])
        history_best_rougeL.append(iter_best_scores_tuple[2]); history_best_bert.append(iter_best_scores_tuple[3])
        history_best_bleu.append(iter_best_scores_tuple[4]); history_best_summarization.append(iter_best_scores_tuple[5])
        history_best_grammar.append(iter_best_scores_tuple[6])
        current_iter_combined_score_for_history = BEST_CANDIDATE_WEIGHT_SUMM * iter_best_scores_tuple[5] + BEST_CANDIDATE_WEIGHT_GRAM * iter_best_scores_tuple[6]
        history_best_combined.append(current_iter_combined_score_for_history)

        current_iter_meta_file.write(f"Iteration {iter_idx+1} - Logged Best Candidate (0.97S+0.03G from F1): {iter_best_candidate_text}\n")
        current_iter_meta_file.write(f"Iteration {iter_idx+1} - Its Objectives (Summ, Gram): {iter_best_objectives}\n")
        current_iter_meta_file.write(f"Iteration {iter_idx+1} - Its Scores (R1,R2,RL,B,BL,S,G): {iter_best_scores_tuple}\n")
        current_iter_meta_file.write(f"Iteration {iter_idx+1} - Its Combined Score (0.97S+0.03G): {current_iter_combined_score_for_history:.2f}\n")
        tqdm.write(f"Iteration {iter_idx+1} - Logged Best Candidate (0.97S+0.03G from F1): {iter_best_candidate_text[:50]}... Objectives: {iter_best_objectives}, Combined (0.97/0.03): {current_iter_combined_score_for_history:.2f}")
        if 'wandb' in globals() and wandb.run:
            wandb.log({"Iteration Logged Best Candidate Text": iter_best_candidate_text,
                       "Iteration Logged Best Summarization Score": iter_best_scores_tuple[5],
                       "Iteration Logged Best Grammar Score": iter_best_scores_tuple[6],
                       "Iteration Logged Best ROUGE-1": iter_best_scores_tuple[0],
                       "Iteration Logged Best ROUGE-2": iter_best_scores_tuple[1],
                       "Iteration Logged Best ROUGE-L": iter_best_scores_tuple[2],
                       "Iteration Logged Best BERT": iter_best_scores_tuple[3],
                       "Iteration Logged Best BLEU": iter_best_scores_tuple[4],
                       "Iteration Logged Best Combined Score (0.97S+0.03G)": current_iter_combined_score_for_history,
                       "API Calls": complete_phi4.count, "Iteration": iter_idx + 1})

        if current_iter_combined_score_for_history > overall_best_combined_score:
            overall_best_combined_score = current_iter_combined_score_for_history
            overall_best_candidate_text = iter_best_candidate_text
            overall_best_candidate_objectives = iter_best_objectives
            patience_counter = 0
            current_iter_meta_file.write(f"New Overall Best Candidate Found (based on 0.97S+0.03G)!\n")
            tqdm.write(f"New Overall Best Candidate Found (based on 0.97S+0.03G)! Score: {overall_best_combined_score:.2f}")
        else: patience_counter += 1
        if patience_counter >= args.patience:
            tqdm.write(f"Patience ({args.patience}) exhausted. Stopping early.")
            current_iter_meta_file.write("Patience exhausted. Stopping early.\n")
            break
    torch.cuda.empty_cache(); gc.collect()
# --- End of Evolutionary Loop ---
with open(meta_path, 'a', encoding='utf-8') as final_meta_file:
    if iter_idx < args.num_iter -1 and complete_phi4.count < args.budget and patience_counter < args.patience:
        pass
    elif 'current_population_data_for_sorting' in locals() and 'fronts' in locals() and fronts and fronts[0]:
         plot_pareto_front(current_population_data_for_sorting, fronts, iter_idx, args.meta_dir, final=True)
    else: tqdm.write("Could not plot final Pareto front (no data or fronts).")

    # --- Save predictions for the best candidate from the final iteration on the training set ---
    final_meta_file.write("\nSaving training set predictions for the best candidate from the final iteration...\n")
    tqdm.write("\nSaving training set predictions for the best candidate from the final iteration...")
    if 'iter_best_candidate_text' in locals() and iter_best_candidate_text != "N/A":
        best_cand_to_save = iter_best_candidate_text
        tqdm.write(f"Candidate for saving: {best_cand_to_save}")
        (train_preds, train_r1, train_r2, train_rL, train_bert, train_bleu,
         train_answers, train_indices, train_r1_list, train_r2_list,
         train_rL_list, train_bert_list, train_bleu_list) = run(
            mode=args.mode, batch_size=args.batch_size, num_shots=args.num_shots,
            chosen_task_name=chosen_task_name,
            num_samples=num_train_samples,
            data_seed=args.train_seed,
            override_prompts=True, function_to_call=custom_urdu_instruction_prompt,
            split='train', modified={"Definition": best_cand_to_save}, args=args)
        pname_train = args.meta_name.split(".")[0] + "_final_iter_train_predictions.json"
        detailed_predictions_output_train = []
        for i in range(len(train_preds)):
            detailed_predictions_output_train.append({
                "id": train_indices[i] if i < len(train_indices) else None,
                "reference_summary": train_answers[i] if i < len(train_answers) else None,
                "generated_summary": train_preds[i] if i < len(train_preds) else None,
                "rouge1": train_r1_list[i] if i < len(train_r1_list) else 0.0,
                "rouge2": train_r2_list[i] if i < len(train_r2_list) else 0.0,
                "rougeL": train_rL_list[i] if i < len(train_rL_list) else 0.0,
                "bert_score": train_bert_list[i] if i < len(train_bert_list) else 0.0,
                "bleu_score": train_bleu_list[i] if i < len(train_bleu_list) else 0.0,
            })
        pred_dump_train = {
            "final_iteration_best_prompt": best_cand_to_save,
            "overall_avg_rouge1_train": train_r1,
            "overall_avg_rouge2_train": train_r2,
            "overall_avg_rougeL_train": train_rL,
            "overall_avg_bert_train": train_bert,
            "overall_avg_bleu_train": train_bleu,
            "predictions_detailed_on_train_set": detailed_predictions_output_train
        }
        ppath_train = os.path.join(args.meta_dir, pname_train)
        with open(ppath_train, "w+", encoding="utf-8") as pfile_train:
            json.dump(pred_dump_train, pfile_train, ensure_ascii=False, indent=2)
        tqdm.write(f"Saved training predictions to {ppath_train}")
        if 'wandb' in globals() and wandb.run:
            wandb.save(ppath_train)
    else:
        tqdm.write("Could not save training predictions, best candidate from final iteration not available.")
        final_meta_file.write("Could not save training predictions, best candidate from final iteration not available.\n")
    # --- End of saving predictions ---

    final_meta_file.write(f"\nSearch Finished.\nAPICalls for search: {complete_phi4.count}\n")
    tqdm.write(f"APICalls for search: {complete_phi4.count}")
    if 'wandb' in globals() and wandb.run: wandb.log({"Total API Calls (Search)": complete_phi4.count})
    plot_best_candidate_scores(args.meta_dir, history_best_rouge1, history_best_rouge2, history_best_rougeL,
                               history_best_bert, history_best_bleu, history_best_summarization,
                               history_best_grammar, history_best_combined)



    # tqdm.write("\nTesting the overall best candidate found...")
    # final_meta_file.write("\nTesting the overall best candidate found...\n")
    # final_s_score, final_r1_score, final_r2_score, final_rL_score, final_b_score, final_bl_score = score_final(
    #     overall_best_candidate_text, 'test', write=args.write_preds)
    # tqdm.write(f"Final Overall Best Candidate (based on 0.97S+0.03G): {overall_best_candidate_text}")
    # tqdm.write(f"Final Test Scores (Summarization, ROUGE-1, ROUGE-2, ROUGE-L, BERT, BLEU): ({final_s_score:.2f}, {final_r1_score:.2f}, {final_r2_score:.2f}, {final_rL_score:.2f}, {final_b_score:.2f}, {final_bl_score:.2f})")
    # final_meta_file.write(f"Final Overall Best Candidate (based on 0.97S+0.03G): {overall_best_candidate_text}\n")
    # final_meta_file.write(f"Final Test Summarization Score: {final_s_score:.2f}\n")
    # final_meta_file.write(f"Final Test ROUGE-1 Score: {final_r1_score:.2f}\n")
    # final_meta_file.write(f"Final Test ROUGE-2 Score: {final_r2_score:.2f}\n")
    # final_meta_file.write(f"Final Test ROUGE-L Score: {final_rL_score:.2f}\n")
    # final_meta_file.write(f"Final Test BERT Score: {final_b_score:.2f}\n")
    # final_meta_file.write(f"Final Test BLEU Score: {final_bl_score:.2f}\n")
    # if 'wandb' in globals() and wandb.run:
    #     wandb.log({"Final Overall Best Candidate Text": overall_best_candidate_text, "Final Test Summarization Score": final_s_score,
    #                "Final Test ROUGE-1 Score": final_r1_score, "Final Test ROUGE-2 Score": final_r2_score,
    #                "Final Test ROUGE-L Score": final_rL_score, "Final Test BERT Score": final_b_score,
    #                "Final Test BLEU Score": final_bl_score})
    # end_time = time.time(); total_time = end_time - start_time
    tqdm.write(f"Total execution time: {total_time:.2f} seconds")
    final_meta_file.write(f"Total execution time: {total_time:.2f} seconds\n")
    if 'wandb' in globals() and wandb.run:
        wandb.log({"Total Execution Time (seconds)": total_time})
        wandb.save(os.path.join(args.meta_dir, args.meta_name))
        if args.write_preds:
            pname = args.meta_name.split('.')[0] + "_predictions.json"
            ppath = os.path.join(args.meta_dir, pname)
            if os.path.exists(ppath): wandb.save(ppath)

global_progress_bar.close()
if 'wandb' in globals() and wandb.run:
    wandb.finish()
if 'meta_file_initial_open' in globals() and meta_file_initial_open and not meta_file_initial_open.closed:
    meta_file_initial_open.close()